# **Training Pipeline trên TPU v5e (Google Colab)**

Quy trình huấn luyện seq2seq cho tóm tắt văn bản tiếng Việt trên **TPU v5e-1**.

### Lưu ý quan trọng khi dùng TPU:
- Dùng **bf16** (TPU v5e hỗ trợ native bfloat16)
- **Không dùng** CUDA quantization (BitsAndBytes không hỗ trợ TPU)
- Cần cài **torch_xla** để PyTorch giao tiếp với TPU
- HuggingFace Trainer tự động detect TPU khi có torch_xla

---
## 1. Kiểm tra TPU & Cài đặt

In [1]:
# Kiểm tra TPU có sẵn không
import os

# Kiểm tra biến môi trường TPU
tpu_name = os.environ.get('TPU_NAME', os.environ.get('COLAB_TPU_ADDR', None))
print(f"TPU_NAME: {tpu_name}")

# Kiểm tra torch_xla
try:
    import torch_xla
    import torch_xla.core.xla_model as xm

    device = xm.xla_device()
    print(f"✅ TPU sẵn sàng! Device: {device}")
    print(f"   torch_xla version: {torch_xla.__version__}")
except Exception as e:
    print(f"❌ Không tìm thấy TPU: {e}")
    print("Hãy chắc chắn đã chọn Runtime > Change runtime type > TPU v5e")

TPU_NAME: None
❌ Không tìm thấy TPU: No module named 'torch_xla'
Hãy chắc chắn đã chọn Runtime > Change runtime type > TPU v5e


In [2]:
# Clone repo & chuyển vào thư mục project
import os

REPO_URL = "https://github.com/dungcony/sumarization.git"
REPO_DIR = os.path.join(os.getcwd(), "sumarization")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 33 (delta 18), reused 32 (delta 17), pack-reused 0 (from 0)
Unpacking objects: 100% (33/33), 39.67 MiB | 4.55 MiB/s, done.
From https://github.com/dungcony/sumarization
   ab12583..9ce66d6  master     -> origin/master
Updating ab12583..9ce66d6
Updating files: 100% (20/20), done.
Fast-forward
 .gitignore                                         |     1 -
 .idea/modules.xml                                  |     2 +-
 ...vn-summarization@1.iml => vn-summarization.iml} |     2 +-
 configs/bartpho.yaml                               |     4 +-
 configs/mt5_base.yaml                              |     4 +-
 configs/vit5_base.yaml                             |     4 +-
 configs/vit5_base_lora.yaml                        |     4 +-
 configs/vit5_warmstart.yaml                        |     4 +-
 data/fine-turn/Kmeans_1024_new.csv                 | 17

In [3]:
# Cài đặt dependencies
# Colab đã có sẵn torch, torch_xla, transformers, datasets
# Chỉ cần cài thêm những gì thiếu
%pip install -e . 2>&1 | tail -5

  Attempting uninstall: vn-summarization
    Found existing installation: vn-summarization 1.0.0
    Uninstalling vn-summarization-1.0.0:
      Successfully uninstalled vn-summarization-1.0.0


---
## 2. Import

In [4]:
from __future__ import annotations

import time
from pathlib import Path

from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from src.config import load_config, apply_overrides, config_to_dict
from src.data import load_and_preprocess
from src.evaluator import build_compute_metrics
from src.model import (
    apply_lora,
    enable_gradient_checkpointing,
    freeze_encoder,
    load_model,
    load_tokenizer,
)
from src.utils import (
    format_duration,
    format_number,
    save_json,
    set_seed,
    setup_logger,
    count_parameters,
)

logger = setup_logger("notebook")
print("✅ Import thành công!")

✅ Import thành công!


---
## 3. Cấu hình phần cứng (đọc phần cứng để áp dụng gput hoặc tpu)

In [5]:
# ============================================================
# Tải config YAML & ghi đè cấu hình động (GPU/TPU)
# ============================================================
import os

# Chọn config cơ sở
CONFIG_FILE = "configs/vit5_base.yaml"

config = load_config(CONFIG_FILE)

# Tự động phát hiện đang dùng TPU hay GPU
is_tpu = "TPU_NAME" in os.environ or "COLAB_TPU_ADDR" in os.environ

# Ghi đè các thiết lập
config = apply_overrides(config, {
    # Nếu là TPU thì ép dùng bf16, nếu GPU thì để auto (tự động chọn fp16)
    "training.precision": "bf16" if is_tpu else "auto",

    # TPU xài adafactor tiết kiệm RAM, GPU xài adamw_torch nhanh hơn
    "training.optim": "adafactor" if is_tpu else "adamw_torch",

    "training.output_dir": "/content/outputs/vit5_base_model" if not os.path.exists(
        "/kaggle") else "/kaggle/working/outputs/vit5_base_model",

    # Batch size (T4 GPU thường yếu hơn TPU v5e một chút nên set batch=4 cho an toàn)
    "training.per_device_train_batch_size": 8 if is_tpu else 4,
    "training.per_device_eval_batch_size": 16 if is_tpu else 8,
    "training.gradient_accumulation_steps": 1 if is_tpu else 2,  # Bù lại batch size nhỏ cho GPU

    "data.max_train_samples": 200,  # Chỉ lấy 200 bài báo để học thử
    "data.max_eval_samples": 50,  # Chỉ lấy 50 bài báo để chấm điểm thử
    "training.max_steps": 20,  # Chỉ chạy 20 bước rồi dừng luôn
})

print(f"Hardware:   {'TPU' if is_tpu else 'GPU'}")
print(f"Model:      {config.model.name_or_path}")
print(f"Precision:  {config.training.precision}")
print(f"Batch size: {config.training.per_device_train_batch_size}")
print(f"Optimizer:  {config.training.optim}")
print(f"Output:     {config.training.output_dir}")


Hardware:   GPU
Model:      VietAI/vit5-base
Precision:  auto
Batch size: 4
Optimizer:  adamw_torch
Output:     /kaggle/working/outputs/vit5_base_model


---
## 4. Tải Model & Data

In [6]:
# Thiết lập seed
set_seed(config.training.seed)

# Tải tokenizer
tokenizer = load_tokenizer(config.model)
print(f"✅ Tokenizer: vocab_size={tokenizer.vocab_size}")

[INFO] src.model: Đang tải T5 SentencePiece tokenizer cho: VietAI/vit5-base
INFO:src.model:Đang tải T5 SentencePiece tokenizer cho: VietAI/vit5-base
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[INFO] src.model: Đã tải tokenizer: vocab_size=36000, type=T5Tokenizer
INFO:src.model:Đã tải tokenizer: vocab_size=36000, type=T5Tokenizer


✅ Tokenizer: vocab_size=36000


In [7]:
# Tải model
model = load_model(config.model, tokenizer, config.generation)

# Áp dụng gradient checkpointing (tiết kiệm bộ nhớ TPU)
if config.training.gradient_checkpointing:
    enable_gradient_checkpointing(model)

# Đóng băng encoder nếu cần
if config.training.freeze_encoder:
    freeze_encoder(model)

# Áp dụng LoRA nếu bật
model = apply_lora(model, config.lora)

params = count_parameters(model)
print(f"✅ Model loaded: {format_number(params['total'])} total, "
      f"{format_number(params['trainable'])} trainable ({params['trainable_percent']}%)")

[INFO] src.model: Đang tải mô hình: VietAI/vit5-base
INFO:src.model:Đang tải mô hình: VietAI/vit5-base
[INFO] src.model: Đã tải mô hình: 225,950,976 tổng số tham số, 225,950,976 có thể huấn luyện (100.0%)
INFO:src.model:Đã tải mô hình: 225,950,976 tổng số tham số, 225,950,976 có thể huấn luyện (100.0%)
[INFO] src.model: LoRA bị vô hiệu hóa, sử dụng fine-tuning toàn bộ (full fine-tuning)
INFO:src.model:LoRA bị vô hiệu hóa, sử dụng fine-tuning toàn bộ (full fine-tuning)


✅ Model loaded: 225,950,976 total, 225,950,976 trainable (100.0%)


In [8]:
# Tải & tiền xử lý dữ liệu
datasets = load_and_preprocess(tokenizer, config.data)

print(f"✅ Train:      {len(datasets['train'])} samples")
print(f"✅ Validation: {len(datasets['validation'])} samples")

[INFO] src.data: Đang tải dữ liệu huấn luyện từ: data/fine-turn/train-00000-of-00001.parquet
INFO:src.data:Đang tải dữ liệu huấn luyện từ: data/fine-turn/train-00000-of-00001.parquet
[INFO] src.data: Đang tải dữ liệu đánh giá từ: data/fine-turn/valid-00000-of-00001.parquet
INFO:src.data:Đang tải dữ liệu đánh giá từ: data/fine-turn/valid-00000-of-00001.parquet


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

[INFO] src.data: Đã tải tập dữ liệu: 10775 train, 1349 validation
INFO:src.data:Đã tải tập dữ liệu: 10775 train, 1349 validation
[INFO] src.data: Đang giới hạn phần train xuống còn 200 mẫu
INFO:src.data:Đang giới hạn phần train xuống còn 200 mẫu


Tokenizing train:   0%|          | 0/200 [00:00<?, ? examples/s]

[INFO] src.data: train: đã tokenize 200 mẫu
INFO:src.data:train: đã tokenize 200 mẫu
[INFO] src.data: Đang giới hạn phần validation xuống còn 50 mẫu
INFO:src.data:Đang giới hạn phần validation xuống còn 50 mẫu


Tokenizing validation:   0%|          | 0/50 [00:00<?, ? examples/s]

[INFO] src.data: validation: đã tokenize 50 mẫu
INFO:src.data:validation: đã tokenize 50 mẫu


✅ Train:      200 samples
✅ Validation: 50 samples


---
## 5. Cấu hình Trainer cho TPU

In [9]:
tc = config.training
output_dir = Path(tc.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# Xác định precision
fp16 = tc.precision == "fp16"
bf16 = tc.precision == "bf16"

training_args = Seq2SeqTrainingArguments(
    output_dir=tc.output_dir,
    seed=tc.seed,

    # Epochs & steps
    num_train_epochs=tc.num_train_epochs,
    max_steps=tc.max_steps,

    # Batch size
    per_device_train_batch_size=tc.per_device_train_batch_size,
    per_device_eval_batch_size=tc.per_device_eval_batch_size,
    gradient_accumulation_steps=tc.gradient_accumulation_steps,

    # Optimizer
    learning_rate=tc.learning_rate,
    weight_decay=tc.weight_decay,
    warmup_ratio=tc.warmup_ratio,
    lr_scheduler_type=tc.lr_scheduler_type,
    optim=tc.optim,  #thuật toán cập nhật kiến thức

    # Regularization
    label_smoothing_factor=tc.label_smoothing_factor,

    # Precision — TPU v5e dùng bf16
    fp16=fp16,
    bf16=bf16,

    # Evaluation & saving
    eval_strategy=tc.eval_strategy,  #số bước học cho mỗi lần kiểm tra
    eval_steps=tc.eval_steps,
    save_strategy=tc.save_strategy,  # số bước học cho mỗi lần lưu
    save_steps=tc.save_steps,
    save_total_limit=tc.save_total_limit,  # số bản lưu trữ được giữ lại
    logging_steps=tc.logging_steps,

    # Best model
    metric_for_best_model=tc.metric_for_best_model,
    greater_is_better=tc.greater_is_better,
    load_best_model_at_end=tc.load_best_model_at_end,  #chọn bản tốt nhất để lưu

    # Generation during eval
    predict_with_generate=True,  #yêu cầu tạo ra 1 bài báo và tóm tắt rồi mới chấm điểm
    generation_max_length=config.generation.max_length,  #số token tối đa được viết khi làm bài kiểm tra

    # Logging — dùng tensorboard
    report_to=["tensorboard"],
    logging_dir=str(output_dir / "logs"),

    # ===== TPU-specific =====
    # dataloader_drop_last giúp tránh batch cuối bị lẻ trên TPU
    dataloader_drop_last=True,
)

print(f"✅ TrainingArguments đã sẵn sàng")
print(f"   bf16={bf16}, fp16={fp16}")
print(f"   device: {training_args.device}")

✅ TrainingArguments đã sẵn sàng
   bf16=False, fp16=False
   device: cuda:0


In [10]:
# Data collator (dynamic padding)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

# Hàm tính ROUGE metrics
compute_metrics = build_compute_metrics(tokenizer)

# Callbacks
callbacks = []
if tc.early_stopping_patience > 0:
    callbacks.append(
        EarlyStoppingCallback(
            early_stopping_patience=tc.early_stopping_patience,
        )
    )
    print(f"✅ Early stopping: patience={tc.early_stopping_patience}")

# Xây dựng Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

print("✅ Trainer đã sẵn sàng!")

✅ Early stopping: patience=5


/tmp/ipykernel_4076/687458540.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


✅ Trainer đã sẵn sàng!


---
## 6. Huấn luyện 🚀

In [11]:
print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN")
print("=" * 60)
print(f"  Model:       {config.model.name_or_path}")
print(f"  Epochs:      {tc.num_train_epochs}")
print(f"  Batch size:  {tc.per_device_train_batch_size}")
print(f"  Grad accum:  {tc.gradient_accumulation_steps}")
print(f"  LR:          {tc.learning_rate}")
print(f"  Optimizer:   {tc.optim}")
print(f"  Precision:   {tc.precision}")
print(f"  LoRA:        {config.lora.enabled}")
print("=" * 60)

start_time = time.time()

train_result = trainer.train(
    resume_from_checkpoint=tc.resume_from_checkpoint,
)

elapsed = time.time() - start_time
print(f"\n✅ Huấn luyện hoàn thành! Thời gian: {format_duration(elapsed)}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 0}.


BẮT ĐẦU HUẤN LUYỆN
  Model:       VietAI/vit5-base
  Epochs:      3
  Batch size:  4
  Grad accum:  2
  LR:          3e-05
  Optimizer:   adamw_torch
  Precision:   auto
  LoRA:        False


Step,Training Loss,Validation Loss



✅ Huấn luyện hoàn thành! Thời gian: 56s


---
## 7. Lưu model & Đánh giá

In [12]:
# Lưu model tốt nhất
best_dir = output_dir / "best"
print(f"Đang lưu model tốt nhất tới: {best_dir}")
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
print("✅ Đã lưu model!")

Đang lưu model tốt nhất tới: /kaggle/working/outputs/vit5_base_model/best
✅ Đã lưu model!


In [13]:
# Đánh giá cuối cùng
print("Đang chạy đánh giá cuối cùng...")
eval_results = trainer.evaluate(metric_key_prefix="eval")

print("\n" + "=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ")
print("=" * 60)
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
print("=" * 60)

Đang chạy đánh giá cuối cùng...


ValueError: `decoder_start_token_id` or `bos_token_id` has to be defined for encoder-decoder generation.

In [ ]:
# Lưu metrics & config
train_metrics = train_result.metrics
train_metrics["train_runtime_formatted"] = format_duration(
    train_metrics.get("train_runtime", 0)
)

save_json(train_metrics, output_dir / "train_results.json")
save_json(eval_results, output_dir / "eval_results.json")
save_json(config_to_dict(config), output_dir / "resolved_config.json")

print("✅ Đã lưu tất cả metrics!")
print(f"   📁 {output_dir}")

---
## 8. (Tùy chọn) Copy model lên Google Drive (Chỉ dành cho Colab)
*Lưu ý: Nếu bạn chạy trên Kaggle thì KHÔNG chạy Cell này. Kaggle sẽ tự động đóng gói output để bạn tải về ở menu bên phải.*

In [ ]:
# Mount Google Drive & copy model để không bị mất khi disconnect
from google.colab import drive

drive.mount('/content/drive')

DRIVE_OUTPUT = "/content/drive/MyDrive/summarization_models/vit5_base_tpu"
!mkdir -p {DRIVE_OUTPUT}
!cp -r {output_dir / 'best'}/* {DRIVE_OUTPUT}/
!cp {output_dir}/*.json {DRIVE_OUTPUT}/

print(f"✅ Đã copy model lên Google Drive: {DRIVE_OUTPUT}")

---
## 9. (Tùy chọn) Test thử model

In [ ]:
from src.predict import summarize

test_text = """Thủ tướng Chính phủ vừa phê duyệt đề án phát triển ứng dụng 
trí tuệ nhân tạo tại Việt Nam giai đoạn 2025-2030. Theo đó, Việt Nam đặt 
mục tiêu trở thành một trong những trung tâm đổi mới sáng tạo về AI trong 
khu vực ASEAN. Đề án tập trung vào 5 lĩnh vực ưu tiên gồm y tế, giáo dục, 
nông nghiệp, giao thông và sản xuất công nghiệp."""

summary = summarize(
    text=test_text,
    model_path=str(best_dir),
    config=config,
)

print("📄 Bài gốc:")
print(test_text.strip())
print("\n📝 Tóm tắt:")
print(summary)